In [1]:
import numpy as np
import pandas as pd

In [ ]:
with open(r"StockEtablissement_utf8.csv", 'r', encoding='utf-8') as f:
    print(f.read(500))  # Affiche les 500 premiers caractères

siren,nic,siret,statutDiffusionEtablissement,dateCreationEtablissement,trancheEffectifsEtablissement,anneeEffectifsEtablissement,activitePrincipaleRegistreMetiersEtablissement,dateDernierTraitementEtablissement,etablissementSiege,nombrePeriodesEtablissement,complementAdresseEtablissement,numeroVoieEtablissement,indiceRepetitionEtablissement,dernierNumeroVoieEtablissement,indiceRepetitionDernierNumeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codePostalEtablissement,libelleC


In [5]:
#lecture du doc pour le calcul de flux
df=pd.read_csv("output_icpe_with_NAF.csv",sep=',',encoding='utf-8')
df.head()


,num_etablissement,entete_nom,entete_siret,entete_etat_d_activite,entete_regime_en_vigueur_de_l_etablissement_2,entete_statut_seveso,entete_ied_-_mtd,derniere_date_inspection,situation_administrative_code_rubrique,situation_administrative_alinea,situation_administrative_libelle_rubrique,situation_administrative_regime_autorise,situation_administrative_volume,code_naf
0,39,COLLECTES VALORISATION ENERGIE DECHETS - COVED,3.434035e+13,En exploitation avec titre,Autorisation,Non Seveso,Oui,2026-01-26,2782,NaN,Autres traitements biologiques de déchets non ...,Autorisation,NaN,38.32Z
1,39,COLLECTES VALORISATION ENERGIE DECHETS - COVED,3.434035e+13,En exploitation avec titre,Autorisation,Non Seveso,Oui,2026-01-26,3532,NaN,Valorisation de déchets non dangereux,Autorisation,518.000 t/j,38.32Z
2,2,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,4.229897e+13,En exploitation avec titre,Autorisation,Non Seveso,Non,NaN,1436,2,Liquides combustibles de point éclair compris ...,Déclaration avec contrôle,900.000 t,41.10A
3,2,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,4.229897e+13,En exploitation avec titre,Autorisation,Non Seveso,Non,NaN,1450,1,Solides inflammables,Autorisation,150.000 t,41.10A
4,2,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,4.229897e+13,En exploitation avec titre,Autorisation,Non Seveso,Non,NaN,1510,1,Entrepôts couverts soumis à EE systématique,Autorisation,NaN,41.10A


Les flux sont déjà présents dans des unités différentes (masse, volume, voire surface (?)), et j'ai souvent des flux en tonne que je suppose être des tonnes par jour. J'ai demandé à Marisol qui m'a dit de ne pas faire de conversion masse/volume.Pour l'instant je ne vois pas vraiment ce que je peux faire d'utile.

# Vérification des numéros siret #
Par un merge sur le code postal où l'on vérifie que l'on ne perd pas de lignes.

In [12]:
result=pd.read_csv("result_trie.csv",sep=',',na_values=['[ND]'])
result.head()

#pour éviter les pbms mémoire on va tout convertir en types nullables donc en Int32
#et donc on définit les nan
result['cd_postal']=result['cd_postal'].astype('str').str.strip()

C:\Users\esthe\AppData\Local\Temp\ipykernel_35040\4114534714.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  result=pd.read_csv("result_trie.csv",sep=',',na_values=['[ND]'])


In [ ]:
# Merge sur le siret pour récupérer le code NAF
# ATTENTION : tourne en 1m30 environ
columns_to_use = ['siret', 'activitePrincipaleEtablissement','codePostalEtablissement', 'coordonneeLambertAbscisseEtablissement','coordonneeLambertOrdonneeEtablissement']
dtype = {
    'codePostalEtablissement': 'str',  # ou 'str' si format texte (ex: "75001")
    'coordonneeLambertAbscisseEtablissement': 'Float64',
    'coordonneeLambertOrdonneeEtablissement': 'Float64'
}
siren = pd.read_csv(
    'StockEtablissement_utf8.csv',
    usecols=columns_to_use,
    dtype=dtype,
    sep=',',
    low_memory=True,  # évite l'inférence de type par blocs
    na_values=['[ND]']  # Remplace "[ND]" par NaN
)
siren.rename(columns={'siret': 'entete_siret'}, inplace=True)
siren['entete_siret'] = siren['entete_siret'].astype(str)
result['entete_siret'] = result['entete_siret'].astype(str)

MemoryError: 

In [ ]:
# ATTENTION : tourne en 13m environ
tab = df.merge(df_NAF, on='entete_siret', how='left')
tab.rename(columns={'activitePrincipaleEtablissement': 'code_naf'}, inplace=True)
tab.to_csv("output_icpe_with_NAF.csv", index=False)
print("Le CSV a été nettoyé et enregisté avec succès ! ;)")

In [ ]:
#ATTENTION code remplacé par le code précédent 
# on ouvre la base siren et on extrait les colonnes intéressantes
columns_to_use = ['codePostalEtablissement', 'coordonneeLambertAbscisseEtablissement','coordonneeLambertOrdonneeEtablissement']
#siren = pd.read_csv('StockEtablissement_utf8.csv', usecols=columns_to_use, sep=',')

dtype = {
    'codePostalEtablissement': 'str',  # ou 'str' si format texte (ex: "75001")
    'coordonneeLambertAbscisseEtablissement': 'Float64',
    'coordonneeLambertOrdonneeEtablissement': 'Float64'
}

siren = pd.read_csv(
    'StockEtablissement_utf8.csv',
    usecols=columns_to_use,
    dtype=dtype,
    sep=',',
    low_memory=True,  # évite l'inférence de type par blocs
    na_values=['[ND]']  # Remplace "[ND]" par NaN
)

In [9]:
siren.head()

,codePostalEtablissement,coordonneeLambertAbscisseEtablissement,coordonneeLambertOrdonneeEtablissement
0,98770,<NA>,<NA>
1,84000,851150.098259,6317267.146095
2,84000,848084.236681,6316548.347115
3,13420,913005.278239,6245869.653506
4,13004,894589.970053,6247476.026088


In [4]:
import dask
import pyarrow
print(dask.__version__)  # Doit afficher la version (ex: 2024.5.1)
print(pyarrow.__version__)
print("PyArrow installé ici :", pyarrow.__file__)
print("Version détectée par Python :", pyarrow.__version__)

2026.6.0
24.0.0
PyArrow installé ici : c:\miniconda\Lib\site-packages\pyarrow\__init__.py
Version détectée par Python : 24.0.0


In [ ]:
#tableau_merged=pd.concat(result,siren,left_on='cd_postal',right_on='codePostalEtablissement',sort=False)
import dask.dataframe as dd
ddresult = dd.from_pandas(result, npartitions=20,chunksize=None)
ddsiren = dd.from_pandas(siren, npartitions=20,chunksize=None)
result_tableau = dd.merge(ddresult,ddsiren, left_on='cd_postal',right_on='codePostalEtablissement').compute()

In [11]:
#on vérifie que l'on n'a pas perdu de lignes
lignes_tableau_final=len(result)
print(lignes_tableau_final)

0
